# Training and sampling : end-to-end tutorial

This notebook trains and samples from interpolant-based generative models on a toy problem. The goal is to showcase the functionality offered by the `stix` library. This tutorial is kept simple and minimal on purpose. Its aim is to give an overview of the library, and describe the general API design.

Before you dive into this tutorial, *we strongly recommend you read the [introduction](https://instadeepai.github.io/stix/introduction.html) page* that summarizes the key mathematical foundations on which `stix` is built.

In this notebook, we will focus on the continuous one-sided case, where we interpolate between Gaussian noise and the target data distribution. Examples of two-sided and discrete cases can be found respectively in [tutorial 5. on coupling](5.coupling.ipynb) and [tutorial 6. on discrete models](6.discrete_models.ipynb).

## Structure of the notebook

1. Imports
2. Toy Dataloader
3. Generative Model
    1. Modality Registry
        1. Interpolant
        2. Embedder
    2. Network
    3. The Generative Model
4. Loss Pipeline
5. Training and Sampling


## 0. Installation

We recommend running this notebook in a **fresh virtual environment**. 

Copy the notebook into some new directory. Then, from a terminal, in the new directory containing the notebook (`1.training_and_sampling.ipynb`):
```
python -m venv my_env
source my_env/bin/activate
pip install notebook ipykernel

python -m ipykernel install --user --name my_env --display-name "my_env"

jupyter notebook
```

The next cell installs `stix` from PyPI together with the extra plotting and dataset packages this notebook uses.

In [ ]:
%pip install stix-ml scikit-learn matplotlib seaborn

## 1. Imports

`stix` exposes its building blocks under the following top-level packages:

- `stix.core` — the central methodological components: modality registry, interpolants, embedders, generative models, coupling.

- `stix.nn` — neural-network building blocks (DiT encoder/backbone/decoder, Fourier-feature time encoder, `EncoderBackboneDecoderNetwork` wrapper).

    - **Whilst we provide some neural network implementations in our library, we purposefully leave [`Network`](https://instadeepai.github.io/stix/api_reference/nn/network.html) abstract as we expect huge diversity in architecture.**

- `stix.training` — training loop, loss pipeline, and corresponding utils.

- `stix.sampling` - O/SDE and CTMC solving/sampling and corresponding utils.

In [ ]:
import logging
from functools import partial

import grain
import jax
import jax.numpy as jnp
import jax.random as jr
import matplotlib.pyplot as plt
import optax
from flax import nnx
from jaxtyping import PyTree
from matplotlib.colors import BoundaryNorm

# Embedders bridge raw data space and the network's embedding space.
from stix.core.embedder import IdentityEmbedder, OneHotDiscreteEmbedder

# A predefined, ready-to-use generative model. `VelocityOneSidedGenerativeModel` trains
# the network to predict the velocity and implements the two methods the rest
# of the library relies on: `get_loss` and `get_generator`.
# A tutorial (3.generative_model.ipynb) is provided to show how to build
# your own custom factory models.
from stix.core.gen_model.factory import VelocityOneSidedGenerativeModel

# Pre-baked one-sided flow matching interpolant.
from stix.core.interpolant import FlowMatchingOneSidedInterpolant

# The modality registry: single source of truth for per-modality config.
from stix.core.modality import ModalityRegistry

# The abstract network contract — we subclass it with a small cross-modal MLP below.
# (`stix`` also ships ready-made networks such as `EncoderBackboneDecoderNetwork` and
# the DiT building blocks under `stix.nn`.)
from stix.nn import Network

# Diffrax-based SDE/ODE solver and direction enum (forward / reverse).
# (`stix` provides a ManualSover with fix time steps that can be used to simultanesouly
# sample an SDE/ODE and a CTMC.)
from stix.sampling.solver import Solver, SolverConfig
from stix.sampling.utils import Direction

# Loss machinery, IO/logging, and the step-based training loop.
from stix.training.loss_pipeline import LossPipeline
from stix.training.training_io_handler import LoggingCategory, TrainingIOHandler
from stix.training.training_loggers import log_metrics_to_line
from stix.training.training_loop import TrainingLoop, TrainingLoopConfig
from stix.typing import Batch, RawSourceTargetPair

In [ ]:
stix_logger = logging.getLogger("stix")
stix_logger.setLevel(logging.INFO)

key = jax.random.PRNGKey(0)

## 2. Dataloader

We use the [grain](https://google-grain.readthedocs.io/en/latest/index.html) library for our dataloaders. See [this tutorial](./2.grain_multimodal_dataloading.ipynb) for in-depth advice on how to set up a dataset in `stix`, and how to use grain.

> **Note:** The grain library enables reproducibility and checkpointing of the dataset state to enable resuming training at the same stage.

In this tutorial, we will be using a very minimal function that samples two data modalities from a Gaussian mixture model (GMM):
- `coordinates` — 2D coordinates of the sampled data point in the $(x, y)$ plane.
- `index` — The index corresponding to the mixture component the coordinates were sampled from.

In [ ]:
CORNERS = jnp.array(
    [[-1.0, -1.0], [1.0, -1.0], [-1.0, 1.0], [1.0, 1.0]], dtype=jnp.float32
)


def sample_coord_and_index_gmm(key: jax.Array) -> tuple[jax.Array, jax.Array]:
    """Ingests a key and returns a sample from the GMM.

    Pick a corner, add Gaussian noise around it, then return the index and the 2D coordinate.
    """
    idx_key, noise_key = jax.random.split(key)
    idx = jax.random.randint(idx_key, (), 0, len(CORNERS))
    coordinates = (
        jax.random.normal(noise_key, (2,), dtype=jnp.float32) * 0.2 + CORNERS[idx]
    )

    return coordinates, idx

To form a datasampler all one requires is a single sampling method that yields [`Batch`](https://instadeepai.github.io/stix/api_reference/typing/index.html#stix.typing.data.Batch) objects. `Batch` follows a specific structure and we must be careful to match this. Let's now write a wrapper around the sampler defined above which achieves this. 

* Each modality must be wrapped in a `RawSourceTargetPair`. 
    * You will notice that we have `source=None`. This is because our [`Interpolant`](https://instadeepai.github.io/stix/api_reference/core/interpolant.html) is one-sided, so it has no source distribution and initial variables are determined by the Gaussian noise.
* We must specify whether each of the modalities are discrete via the `is_discrete` field.
    * This is propagated through the codebase to enable specific [`Embedder`](https://instadeepai.github.io/stix/api_reference/core/embedder.html)s, and to catch errors.

In [ ]:
@jax.jit  # One compiled call builds the whole batch! See the pipeline below for more info.
def sample_gmm(batch_indices: jax.Array, key: jax.Array) -> Batch:
    """Draw one GMM sample per index in ``batch_indices`` and wrap them in a ``Batch``.

    `grain` hands us the indices it has shuffled and grouped; vectorising the draw
    over them is our job. ``jax.random.fold_in`` gives each index its own key, so
    element `i` always yields the same sample (reproducibility) and `key` — one per
    stream — is what distinguishes the train and val streams.

    Each modality is wrapped in a ``RawSourceTargetPair`` with ``source=None``
    (one-sided: the source variable is omitted and initial data is given by the Gaussian noise).
    Discrete modalities are one-hot (the codebase convention), and ``is_discrete`` on the ``Batch`` tells
    the registry which modalities are discrete.
    """
    keys = jax.vmap(partial(jax.random.fold_in, key))(batch_indices)
    coordinates, idx = jax.vmap(sample_coord_and_index_gmm)(keys)
    raw_batch = {
        "coordinates": RawSourceTargetPair(target=coordinates, source=None),
        "index": RawSourceTargetPair(
            target=jax.nn.one_hot(idx, len(CORNERS)), source=None
        ),
    }
    return Batch(
        raw_batch=raw_batch,
        is_discrete={"coordinates": False, "index": True},
    )

Finally, we wire our GMM sampler into a grain dataset. The order below is the one grain recommends, and it is the same in every notebook here:

- **seed** — fix the random key of the dataset for reproducibility;
- **shuffle** — shuffles the data;
- **repeat** — transform the dataset of fixed length into an infinite iterator;
- **batch** — groups indices into a batch; `drop_remainder=True` keeps the shape static, so `jax.jit` compiles the step once;
- **map** — applies `sample_gmm` to the batched indices, producing one `Batch`;
- **to_iter_dataset** — hands back the Python iterator we pull batches from.

Two things here are worth noting for your own pipelines:
1. **grain chooses and groups elements; JAX makes the arrays.** grain itself runs on the CPU, so we give it only indices to shuffle, and a single jitted call turns a batch of indices into a batch of data. Doing the draw one sample at a time instead means one round-trip to the accelerator per sample, which for a toy dataset like this one costs more than the training step it feeds.
2. `to_iter_dataset` goes **last**: everything above it supports random access, which is what lets `batch` group by slicing rather than by pulling elements through an iterator one at a time.


In [ ]:
batch_size = 256


def gmm_dataset(seed: int):
    """Helper function to instantiate a grain dataset."""
    return (
        grain.MapDataset.range(int(1e9))
        .seed(seed)
        .shuffle()
        .repeat()
        .batch(batch_size, drop_remainder=True)
        .map(partial(sample_gmm, key=jr.key(seed)))
        .to_iter_dataset()
    )


train_iter = iter(gmm_dataset(seed=0))
validation_iter = iter(gmm_dataset(seed=42))

## 3. Generative Model

The [`GenerativeModel`](https://instadeepai.github.io/stix/api_reference/core/gen_model.html) ties together the interpolant, embedders, and network. It is responsible for defining the standard methods `get_generator` and `get_loss` used at training and sampling time.

Constructing one requires two things:

1. A [`ModalityRegistry`](https://instadeepai.github.io/stix/api_reference/core/modality.html) — the single source of truth for per-modality config, holding one [`Modality`](https://instadeepai.github.io/stix/api_reference/core/modality.html) per data stream and forming a `PyTree`. Each `Modality`'s fields are optional at construction and populated incrementally:
    * `shape`, `is_discrete` — the modality's shape and whether it is discrete (already set for us by `from_batch`).
    * `interpolant` — the [`Interpolant`](https://instadeepai.github.io/stix/api_reference/core/interpolant.html) defines the noising schedule, and lets you recover diffusion or flow matching as special cases. Here we use the pre-defined [`FlowMatchingOneSidedInterpolant`](https://instadeepai.github.io/stix/api_reference/core/interpolant.html#stix.core.interpolant.FlowMatchingOneSidedInterpolant).
    * `embedder` — converts raw data to the embedded space and back; **the interpolation happens in this embedded space**. Can be a no-op via the `IdentityEmbedder`.

2. A [`Network`](https://instadeepai.github.io/stix/api_reference/nn/network.html#stix.nn.Network) — the model trained to predict a given quantity (noise, score, velocity, etc.). Here we hand-roll a small cross-modal MLP (Section 3.2).

### 3.1. Modality registry

`stix` keeps all per-modality configuration in a single [`ModalityRegistry`](https://instadeepai.github.io/stix/api_reference/core/modality.html) — the central configurable object that later components (`generative_model`, `loss_pipeline`, `solver`) all read from as the single source of truth.
In the subsequent cells in this tutorial we will see how to populate the various elements of the `ModalityRegistry`.

The `ModalityRegistry` has three important methods which are used frequently throughout the codebase:
1. **`.set`** — **update a modality field.** Assigns a value (or a per-modality factory) to a chosen field on every `Modality` (optionally narrowed via `filter_fn`). E.g. `modality_registry.set("interpolant", interpolant)` replaces the interpolant on all modalities. By default (`use_deepcopy=True`) each modality gets its own independent copy.
2. **`.map`** — **compute one value per modality.** Applies a function to every `Modality` and returns a `PyTree` of the same structure — reach for this when a modality's value depends on the modality itself (e.g. a shape-dependent encoder). A thin wrapper around [`jax.tree.map`](https://docs.jax.dev/en/latest/_autosummary/jax.tree.map.html); e.g. `modality_registry.map(lambda modality: 0)` returns the same tree with `0` at every leaf.
3. **`.broadcast`** — **share one object across all modalities.** Copies a single object (or a compatible prefix tree) up to the registry's per-modality structure. Unlike `.set`'s per-modality copies, the broadcast object is **shared** — every modality points to the same instance in memory. A thin wrapper around [`jax.tree.broadcast`](https://docs.jax.dev/en/latest/_autosummary/jax.tree.broadcast.html).

First, let's initialise it: `ModalityRegistry.from_batch` builds the skeleton straight from a batch, reading each modality's `shape` from the data and its `is_discrete` flag from `batch.is_discrete`. We fill the remaining fields (interpolant, embedder, ...) in the sections below via `registry.set(...)`.

In [ ]:
batch = next(train_iter)
# Build the registry straight from a batch: `from_batch` reads each modality's
# shape from the (one-hot) data and its discreteness from `batch.is_discrete`.
# The remaining fields (interpolant, embedder) are filled in the sections below.
modality_registry = ModalityRegistry.from_batch(batch)

#### 3.1.1. Interpolant

Here we use a linear one-sided interpolant. The interpolant always acts on **embedded** variables $z$ (raw data is written $x$ and mapped by an embedder — here an identity, so $z_{\mathrm{tgt}} = x_{\mathrm{tgt}}$). Its path has the form:

$z_t = \beta_t z_{\mathrm{tgt}} + \gamma_t \epsilon$,

where $\beta_t, \gamma_t$ are free to choose.

For this tutorial we use the canonical flow matching interpolant ([Lipman et. al. 2022](https://openreview.net/forum?id=PqvMRDCJT9t)), which is a specific instance of a linear one-sided interpolant with

$z_t = t z_{\mathrm{tgt}} + (1-t) \epsilon$,

i.e. $\beta_t = t, \gamma_t = 1-t$.

We use the `.set` method to perform the replacement.

In [ ]:
# We use the pre-defined Flow Matching schedule for the one sided case.
one_sided_interpolant = FlowMatchingOneSidedInterpolant()

# Broadcast the (shared) interpolant onto every modality in the registry.
modality_registry.set("interpolant", one_sided_interpolant)

#### 3.1.2. Embedder

Each modality has an [`Embedder`](https://instadeepai.github.io/stix/api_reference/core/embedder.html) which maps raw data to embedded data, and back.

> **Remember: the interpolation (and therefore the modelling) happens in this embedding space!** 

We use the following embedders:

- [`IdentityEmbedder`](https://instadeepai.github.io/stix/api_reference/core/embedder.html#stix.core.embedder.IdentityEmbedder) — no-op embedder. Here we use it for the continuous modalities.

- [`OneHotDiscreteEmbedder`](https://instadeepai.github.io/stix/api_reference/core/embedder.html#stix.core.embedder.OneHotDiscreteEmbedder) — turn discrete indices into one-hot vectors. In this tutorial, we treat the obtained one-hot vectors as continuous embedded variables. See [tutorial 6.](6.discrete_models.ipynb) for examples of models with discrete embedded variables.

The `Embedder` base class is very generic, but we offer some popular choices including the option of learnable embeddings with [`LearnedDiscreteEmbedder`](https://instadeepai.github.io/stix/api_reference/core/embedder.html#stix.core.embedder.LearnedDiscreteEmbedder).

Here, again, we use the `.set` method in combination with its `filter_fn` to set the discrete and continuous modalities with different embedders.

In [ ]:
# Embedders are shape-dependent, so install them with per-modality factories.
# Two `set` calls partition the modalities on `is_discrete` (exhaustive for a
# bool flag): discrete -> OneHotDiscreteEmbedder, continuous -> IdentityEmbedder.
modality_registry.set(
    field="embedder",
    value=lambda modality: OneHotDiscreteEmbedder(dm_shape=modality.shape),
    is_factory=True,
    filter_fn=lambda modality: modality.is_discrete,
    use_deepcopy=True,
)
modality_registry.set(
    field="embedder",
    value=lambda modality: IdentityEmbedder(dm_shape=modality.shape),
    is_factory=True,
    filter_fn=lambda modality: not modality.is_discrete,
    use_deepcopy=True,
)

# Generator type: the object each modality yields at sampling time. We use
# `VelocityAndScore` so both ODE (velocity) and SDE (velocity + score) samplers work.

### 3.2. Network

Any network plugged into a `GenerativeModel` need only satisfy the [`Network`](https://instadeepai.github.io/stix/api_reference/nn/network.html#stix.nn.Network) contract: it is called as `network(z_t, t, context_data, context_mask, attention_mask)` and returns a per-modality output pytree. **`stix` is not a neural-network library** — the choice of architecture is entirely yours. We *do* ship some ready-made networks as guides (the [`EncoderBackboneDecoderNetwork`](https://instadeepai.github.io/stix/api_reference/nn/network.html#stix.nn.EncoderBackboneDecoderNetwork) wrapper and DiT encoder/backbone/decoder pieces under `stix.nn`), but to keep this first tutorial focused on the *method* rather than the network, we build a tiny one here.

We build a small **cross-modal** MLP: we concatenate every modality (and time) into one vector, fuse it in a shared trunk, then route the fused state back through one output head per modality. Because the trunk sees all modalities at once, `coordinates` and `index` can inform each other. This is the classic *unified backbone + modality-specific heads* pattern, a flat version of stix's own `EncoderBackboneDecoderNetwork`.

First, split off a PRNG key for the network's parameter initialisation.

In [ ]:
key, model_key = jr.split(key)  # network initialisation
rngs = nnx.Rngs(model_key)

Now the network itself. The output heads are sized directly off the `ModalityRegistry`: each modality's embedding width is read from its `embedder`, via `modality_registry.map`.

**Note: we use `.map` (not `.broadcast`) because we want each output head to have its *own* parameters — `.broadcast` would share a single object across modalities.**

In [ ]:
def _is_module_leaf(leaf):
    """Stop pytree traversal at nnx.Module boundaries.

    The per-modality heads are a pytree whose leaves are the head modules;
    without this, `jax.tree.*` would descend into each head's parameters.
    """
    return isinstance(leaf, nnx.Module)


class CrossModalMLPNetwork(Network):
    """Cross-modal MLP: concatenate every modality, fuse in a shared trunk, then
    route the fused state back through one head per modality.

    Unlike a per-modality network, every output depends on *every* input: the
    shared trunk sees all modalities (and time) at once, so `coordinates` and
    `index` can inform each other. This is the classic "unified backbone +
    modality-specific heads" pattern — a flat miniature of stix's own
    `EncoderBackboneDecoderNetwork`.
    """

    def __init__(self, embedding_dims: PyTree[int], hidden_dim: int, rngs: nnx.Rngs):
        """Build a shared trunk over the concatenated modalities, plus one output head per modality."""
        # Shared trunk (the "backbone"): fuses all modalities + time into one
        # representation. This concatenation is where the modalities interact.
        fused_input_dim = sum(jax.tree.leaves(embedding_dims)) + 1  # +1 for time
        self.trunk = nnx.Sequential(
            nnx.Linear(fused_input_dim, hidden_dim, rngs=rngs),
            nnx.silu,
            nnx.Linear(hidden_dim, hidden_dim, rngs=rngs),
            nnx.silu,
        )

        # Per-modality output heads, held as a pytree (nnx.data, not a keyed dict)
        # so they mirror the modality structure and stay trainable under nnx. Each
        # projects the fused representation back to its modality's own dimension.
        self.heads = nnx.data(
            jax.tree.map(
                lambda dim: nnx.Linear(hidden_dim, dim, rngs=rngs), embedding_dims
            )
        )

    def __call__(
        self, z_t, t, context_data=None, context_mask=None, attention_mask=None
    ):
        """Concatenate the modalities (+ time), fuse in the trunk, then route back per modality."""
        # 1. Concatenate every modality (deterministic treedef order) plus a time
        # column into one vector.
        modalities = jax.tree.leaves(z_t)
        batch_shape = modalities[0].shape[:-1]
        t_col = jnp.broadcast_to(jnp.atleast_1d(t), batch_shape + (1,))
        fused_input = jnp.concatenate(modalities + [t_col], axis=-1)

        # 2. Fuse in the shared trunk -> every output now sees every modality.
        fused = self.trunk(fused_input)

        # 3. Route the shared representation back through each modality's head,
        # rebuilding the input pytree structure.
        return jax.tree.map(
            lambda head: head(fused), self.heads, is_leaf=_is_module_leaf
        )

In [ ]:
# One output head per modality, sized off the registry's embedding dims.
one_sided_network = CrossModalMLPNetwork(
    embedding_dims=modality_registry.map(lambda m: m.embedder.embedding_shape[-1]),
    hidden_dim=128,
    rngs=rngs,
)

### 3.3. The Generative Model

With the `ModalityRegistry` (interpolant + embedders) populated and the network built, we instantiate the [`GenerativeModel`](https://instadeepai.github.io/stix/api_reference/core/gen_model.html) that ties them together. It converts the network's raw output into the quantities the rest of the library consumes. Two methods are declared `@abstractmethod`, so every subclass **must** define them:

- **[`get_loss`](https://instadeepai.github.io/stix/api_reference/core/gen_model.html#stix.core.gen_model.GenerativeModel.get_loss)** — the training objective. **This is what decides what the network learns to predict**, and is therefore inextricably linked to the conversion below.
- **[`get_generator`](https://instadeepai.github.io/stix/api_reference/core/gen_model.html#stix.core.gen_model.GenerativeModel.get_generator)** — convert the network output to a per-modality [`Generator`](https://instadeepai.github.io/stix/api_reference/core/generator.html#stix.core.generator.Generator). **Always required**: it drives all sampling. The concrete type is set per modality via the interpolant's inferred `generator_type` — in our case a `VelocityAndScore` generator the can be used for ODE and SDE sampling.

Models that additionally support classifier-style (intrinsic) guidance override one extra method, [`get_guidance_loss`](https://instadeepai.github.io/stix/api_reference/core/gen_model.html#stix.core.gen_model.GenerativeModel.get_guidance_loss) — a scalar loss of the model's own prediction against conditioning data (typically a per-modality condition). The base implementation raises `NotImplementedError`, so override it only if you need intrinsic guidance. See tutorial [4.conditioning_and_guidance](./4.conditioning_and_guidance.ipynb) for a concrete implementation.


To keep this tutorial focused, we use a **predefined** generative model, [`VelocityOneSidedGenerativeModel`](https://instadeepai.github.io/stix/api_reference/core/gen_model.html), imported above from `stix.core.gen_model.factory`. It is a one-sided model whose network output *is* the velocity, and it implements both methods for you:

- **`get_loss`** — an MSE between the network output and the interpolant's *conditional velocity* (the interpolant provides this ground truth directly), reduced across modalities.
- **`get_generator`** — returns a per-modality generator. The velocity is the identity (the network output already *is* the velocity); the score is additionally derived (stochastic continuous interpolants infer `VelocityAndScore`) from the predicted velocity via each modality's interpolant (used for SDE sampling).

**These are inextricably linked**: the loss decides what the network predicts, which in turn fixes how velocity and score are recovered. The conversions depend on the interpolant, so they cannot be generic — `VelocityOneSidedGenerativeModel` bakes in the choices appropriate for a linear one-sided interpolant. A **later tutorial** walks through writing your own factory models (different prediction targets, cross-modality losses, custom conversions).

It subclasses [`GenerativeModel`](https://instadeepai.github.io/stix/api_reference/core/gen_model.html) directly, and its constructor checks that every modality uses a one-sided linear interpolant, since that is what its conversions assume. (The one-sided initial state for sampling is produced by `ModalityRegistry.sample_initial_state`, rather than a method on the model.)


In [ ]:
one_sided_gen_model = VelocityOneSidedGenerativeModel(
    network=one_sided_network,
    modality_registry=modality_registry,
)

## 4. Loss pipeline

The [`LossPipeline`](https://instadeepai.github.io/stix/api_reference/training/loss_pipeline.html) computes the per-batch training loss. For each batch it:

1. embeds the raw batch via the generative model;
2. samples noise in embedding space and a per-sample time $t \in [0, 1]$ (here with *uniform sampling*);
3. optionally couples source and target samples (not used here);
4. interpolates $z_t$ and runs the network forward at $(z_t, t)$;
5. compares the network output to the ground-truth target quantity.

We will use the default [`TimeSampler`](https://instadeepai.github.io/stix/api_reference/training/time_sampler.html) (uniform distribution on $[0,1]$) and no coupling. See [tutorial 5.coupling](./5.coupling.ipynb) for examples with non-trivial couplings.

In [ ]:
loss_pipeline = LossPipeline()

## 5. Train and sample (one-sided)

With all of our main objects instantiated, we can define the latest ones required to run the training:

- **Optimizer**: Optax optimizer.

- [**IO Handler**](https://instadeepai.github.io/stix/api_reference/training/io_handling.html#stix.training.training_io_handler.TrainingIOHandler): The IO handler that takes care of the logging and checkpointing.

- [**Training Loop**](https://instadeepai.github.io/stix/api_reference/training/training_loop.html#stix.training.training_loop.TrainingLoop): It pulls batches from the data iterators for `num_steps`, keeps an EMA copy of the parameters (used for evaluation and sampling).

In [ ]:
#  We first define a simple optimizer
LEARNING_RATE = 1e-4
optimizer_tx = optax.adam(LEARNING_RATE)

In [ ]:
def make_history_io_handler(
    verbose: bool = True,
) -> tuple[list[dict], TrainingIOHandler]:
    """Build a TrainingIOHandler that records a plottable training history.

    Returns a (history, io_handler) pair: pass io_handler to TrainingLoop, and
    history fills in place with {"step", "train_loss", "eval_loss"} records as
    training runs. With verbose=True, metrics are also printed to the console.
    """
    history: list[dict] = []
    by_step: dict[int, dict] = {}

    def _collect(category: LoggingCategory, to_log: dict, step: int) -> None:
        entry = by_step.get(step)
        if entry is None:
            entry = {"step": step}
            by_step[step] = entry
            history.append(entry)
        if category == LoggingCategory.TRAIN_METRICS:
            entry["train_loss"] = to_log["loss"]
        elif category == LoggingCategory.EVAL_METRICS:
            entry["eval_loss"] = to_log["loss"]

    io_handler = TrainingIOHandler()
    io_handler.attach_logger(_collect)
    if verbose:
        io_handler.attach_logger(log_metrics_to_line)
    return history, io_handler


logs_history, io_handler = make_history_io_handler()

In [ ]:
NUM_STEPS = 10000
EVAL_EVERY_N_STEPS = 100

training_loop_cfg = TrainingLoopConfig(
    num_steps=NUM_STEPS,
    eval_every_n_steps=EVAL_EVERY_N_STEPS,
)

training_loop = TrainingLoop(
    train_data=train_iter,
    val_data=validation_iter,
    loss_pipeline=loss_pipeline,
    gen_model=one_sided_gen_model,
    optimizer_tx=optimizer_tx,
    config=training_loop_cfg,
    io_handler=io_handler,
)
training_loop.run()

In [ ]:
# Plot the train/eval loss curves collected in `logs_history`. Missing keys ->
# nan so the line breaks (the eval-at-start entry has no train_loss; 0 would
# render as -inf on the log axis).
steps = [m["step"] for m in logs_history]
train_losses = [m.get("train_loss", float("nan")) for m in logs_history]
eval_losses = [m.get("eval_loss", float("nan")) for m in logs_history]

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(steps, train_losses, marker="o", label="Train loss")
ax.plot(steps, eval_losses, marker="o", label="Eval loss")
ax.set_yscale("log")
ax.set_title("One-sided: Gaussian Noise -> Categorical GMM Ring")
ax.set_xlabel("Step")
ax.set_ylabel("Loss (log scale)")
ax.legend()
plt.tight_layout()
plt.show()

The [**`Solver`**](https://instadeepai.github.io/stix/api_reference/sampling/solver.html#stix.sampling.solver.Solver) integrates the learned drift along $t \in [0, 1]$ with [`diffrax`](https://docs.kidger.site/diffrax/). To run the solver, the parameters you need to define are the following : 

- **`num_samples`** : The number of samples you want to sample at once, i.e. the batch size of the solver. You could run it multiple times with a different seed to get more samples.

- **`max_solver_steps`** : The maximum number of steps allowed for the `diffrax` solver. This corresponds to a maximum compute budget. `diffrax` doesn't necessarily use the maximum budget allowed. 

- **`stochasticity_scale`** : The stochasticity scale controls the amount of noise injected at sampling time. When set to 0, the solver samples from the corresponding ODE. Read the [stochastic interpolants paper](https://www.jmlr.org/papers/v26/23-1605.html) for more information.

We are integrating forward in time to $t = 1$ with the solver.


In [ ]:
MAX_SOLVER_STEPS = 1000
STOCHASTICITY_SCALE = 1.0

trained_ema_model = training_loop.ema_model


# The function defined here, setting it proportional to \gamma(t), corrects
# for unfavourable asymptotic behavior of the interpolant of choice.
def _stochasticity_scale_per_modality(modality, t):
    """Per modality stochasticity-scale schedule: this modality's gamma(t) times the chosen constant."""
    return modality.interpolant.gamma_fn(t) * STOCHASTICITY_SCALE


# Evaluate the above function for all modalities then return a callable with only the ``t`` argument.
stochasticity_scale_fn = modality_registry.map(
    lambda modality: partial(_stochasticity_scale_per_modality, modality)
)

solver = Solver(
    SolverConfig(
        stochasticity_scale=stochasticity_scale_fn,
        direction=Direction.FORWARD,
        rtol=1e-3,
        atol=1e-3,
        max_steps=MAX_SOLVER_STEPS,
    )
)

To integrate **forward** in time ($t = 0 \to 1$), we need a starting state $z_0$. 

As discussed in the [introduction](https://instadeepai.github.io/stix/introduction.html), at $t=0$, the interpolant function might still depend on target variables. Since these are not available at sampling time, each modality's [`Interpolant`](https://instadeepai.github.io/stix/api_reference/core/interpolant.html) implements a `sample_initial_state` method that computes $z_{t=0}$ based solely on the noise $\epsilon$ and the source $z_{\mathrm{src}}$ (when available). The `ModalityRegistry.sample_initial_state` helper relies on this method to compute $z_0$ for each modality in the registry.

For a one-sided interpolant, the initial state is implemented as
$$
z_0 = \gamma(0)\,\epsilon, \qquad \epsilon \sim \mathcal{N}(0, I)
$$,
which amount to assume $\beta_0 \simeq 0$. 

In [ ]:
NUM_SAMPLES = 256  # Batch size of the solver.
key, key_init, key_solve = jr.split(key, 3)
z_init = modality_registry.sample_initial_state(key_init, num_samples=NUM_SAMPLES)

In [ ]:
# Run the solver independently on each sample (one PRNG key per sample).
def sample_fn(x, k):
    """Solve a single sample forward with its own PRNG key."""
    return solver(trained_ema_model, x, k)


solve_keys = jr.split(key_solve, NUM_SAMPLES)
# `Solver.__call__` decodes back to raw data space internally, so these are
# already raw samples.
raw_samples = jax.vmap(sample_fn)(z_init, solve_keys)

The solver returns data in the raw space. We have

- **`"coordinates"`** (continuous) → 2D points in the plane.
- **`"index"`** (discrete) → per-mode probabilities; we take the `argmax` to recover each point's predicted mode.

In [ ]:
coords = raw_samples["coordinates"]  # (NUM_SAMPLES, 2) points
num_modes = len(CORNERS)  # number of GMM modes
# Discrete modalities decode straight to label indices, so this is already the
# per-sample predicted mode.
pred_modes = raw_samples["index"]

Finally let's plot the generated points, coloured by predicted mode, over a grey cloud of true validation samples for visual comparison.

In [ ]:
# A batch of true samples for visual comparison.
ref_coords = next(validation_iter).raw_batch["coordinates"].target

# Plotting
cmap = plt.get_cmap("tab10", num_modes)
norm = BoundaryNorm([i - 0.5 for i in range(num_modes + 1)], num_modes)

fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(
    ref_coords[:, 0], ref_coords[:, 1], s=12, c="lightgrey", label=r"$\rho_1$ (true)"
)
points = ax.scatter(
    coords[:, 0],
    coords[:, 1],
    s=8,
    c=pred_modes,
    cmap=cmap,
    norm=norm,
    alpha=0.5,
    label="generated",
)

ax.set_title("Generated GMM samples (coloured by predicted mode)")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_aspect("equal")
ax.legend(loc="upper right")
fig.colorbar(points, ax=ax, label="predicted mode", ticks=range(num_modes))
plt.tight_layout()
plt.show()